# Fase 2 · Pipeline MLflow NYC311
Magíster en Data Science UDD · Análisis de Big Data · Profesor Luis Castillo

Este notebook presenta la ejecución real de los cuatro experimentos y el protocolo temporal. Los resultados se leen de MLflow y de artefactos generados por Spark. Las pruebas artificiales previas se conservan separadas. La lectura de evidencias no vuelve a entrenar ni a seleccionar según 2024.

El objetivo es detectar al ingreso solicitudes con duración mayor que el p75 histórico de su tipo, **sin interpretarlo como SLA oficial**. El snapshot no conserva versiones de todos los atributos y la evaluación solo cubre solicitudes con cierre observado y umbral válido.

## 1. Configuración y estado de ejecución

In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd()
CONFIG=json.loads((ROOT/'config.complete.json').read_text())
RUN=ROOT/CONFIG['output_root']; E=RUN/'evidence'
EJECUTAR_DESDE_CERO=False
pd.set_option('display.max_colwidth',75)
pd.set_option('display.float_format',lambda x:f'{x:.4f}')
def read(name): return json.loads((E/name).read_text())
display(read('final_result.json'))

## 2. Entorno reproducible y procedencia
La descarga automática de iCloud también había expulsado librerías de `.venv`. La ejecución se estabilizó en `/private/tmp`, con las mismas versiones fijadas, y los CSV se copiaron verificando SHA256. El contenido original no fue modificado. Para reproducir, usar una carpeta local no sincronizada y seguir README.

In [ ]:
env=read('environment.json')
display(pd.DataFrame(env['libraries'].items(),columns=['biblioteca','version']))
print('Spark:',env['spark'],'| CPU lógicas:',env['cpu_count'],'| RAM física GiB:',env['memory_total']/2**30)
manifest=json.loads((RUN/'bronze/manifest.json').read_text())
display(pd.DataFrame(manifest)[['path','bytes','sha256']])
print('Las rutas de la ejecución temporal quedan documentadas; config.complete.json señala la fuente original para reproducir.')

## 3. Código y reconstrucción opcional
Bronze registra los originales; Silver valida esquema, fechas, llaves y calidad; Gold base conserva variables deterministas y motivos de no elegibilidad. No se imputan fechas ni etiquetas. `prepare_silver` permite reutilización literal de fase 1, pero esta ejecución reconstruyó desde CSV para fijar Nueva York. La preparación estadística se aprende dentro de Pipeline.

In [ ]:
from nyc311_phase2 import setup,session,environment,prepare,tune,benchmark,final_fit,export_runs,CATS,NUMS,candidates
if EJECUTAR_DESDE_CERO:
    CONFIG=json.loads((ROOT/'config.reproduce.json').read_text())
    RUN=setup(CONFIG); E=RUN/'evidence'
    spark=session(CONFIG); spark.sparkContext.setLogLevel('ERROR')
    environment(spark,CONFIG,RUN)
    prepare(spark,CONFIG,RUN)
else:
    print('Modo evidencia: no se modifica la decisión ni se vuelve a utilizar 2024 para selección.')

## 4. Calidad completa y discrepancia temporal
Los conteos globales se reconfirman con los cuatro CSV. La diferencia en negativos frente a fase 1 se explica mediante una recomputación explícita en Santiago y Nueva York. Los timestamps no incluyen offset, por lo que persiste ambigüedad en cambios estacionales. Las fechas recogidas en Python pueden mostrarse en la zona del sistema; el resumen corregido formatea las fechas en Spark, en Nueva York.

In [ ]:
quality=read('quality_summary.json'); display(quality)
display(pd.read_csv(E/'quality_by_year.csv'))
display(read('timezone_reconciliation.json'))
assert quality['n']==quality['unique_keys']==13072334
profile=read('raw_profile.json')
display(pd.DataFrame([{'variable':c,'nulos':profile[c+'__null'],'blancos':profile[c+'__blank'],'cardinalidad_aproximada':profile[c+'__approx_distinct']} for c in ['complaint_type','descriptor','borough','location_type','open_data_channel_type','due_date']]))

## 5. Etiqueta y cortes crecientes
Entrenar 2021 → validar 2022; entrenar 2021–2022 → validar 2023. Solo cierres anteriores al corte pueden entrenar. Las etiquetas de validación deben conocerse antes de 2024. La prueba 2024 se observa antes de 2026-09-11 00:00 Nueva York, aproximación conservadora al día de descarga. Se exigen 100 casos por tipo; raros y nuevos quedan sin etiqueta. Los umbrales se estiman con toda la historia elegible antes del muestreo de desarrollo del 0,2%.

In [ ]:
cohorts=[read('cohort_2022_sample.json'),read('cohort_2023_sample.json')]
final=read('final_result.json'); cohorts.append(final['cohort'])
display(pd.DataFrame(cohorts)[['fold','train_created','train_eligible','train_late_close','train_no_threshold','train_used','eval_created','eval_eligible','eval_no_threshold','eval_used']])
for c in cohorts:
    assert c['train_created']==c['train_eligible']+c['train_late_close']+c['train_invalid_or_open']
    assert c['eval_created']==c['eval_eligible']+c['eval_late_close']+c['eval_invalid_or_open']
print('Conciliaciones de cohortes correctas. Las muestras de desarrollo no sustituyen los conteos completos.')

## 6. Variables y prevención de fuga
Tipo, canal, borough, tipo de ubicación y seis variables cíclicas de creación. Agencia se excluye por reasignaciones; descriptor solo entra en E4. Se excluyen identificador, cierre, duración, status, resolución, actualizaciones y geografía precisa del vector de predictores. Tipo y descriptor pueden ser corregidos posteriormente: se declara esa limitación, sin afirmar una reconstrucción perfecta del ingreso. StringIndexer conserva categorías nuevas; faltantes usan `[MISSING]`; no se añade imputación numérica innecesaria.

In [ ]:
display(pd.DataFrame({'categorias':CATS}))
print('Variables temporales:',NUMS)
display(pd.DataFrame(candidates(),columns=['familia','candidato','parametros']))

## 7. Cuatro experimentos y tuning temporal
E1 base L2 0,01 (grilla también 0,03), E2 L2 0,1 y 0,3, E3 GBT profundidad {3,5} e iteraciones {10,20}, E4 E1 base más MiniLM. ParamGridBuilder define nueve candidatos; el bucle temporal evalúa dos cortes. CrossValidator aleatorio mezclaría pasado y futuro. Se elige área PR media por año, con desempate por costo. El umbral 0,1–0,9 maximiza F1 medio de validación; no representa una capacidad operativa acordada.

In [ ]:
if EJECUTAR_DESDE_CERO: tune(spark,CONFIG,RUN)
import mlflow
mlflow.set_tracking_uri('sqlite:///'+str((RUN/'mlflow.db').resolve()))
runs=mlflow.search_runs(experiment_names=['NYC311_Fase2'])
runs.to_csv(E/'mlflow_runs.csv',index=False)
exp=runs[(runs['tags.kind']=='tuning_candidate')&(runs.status=='FINISHED')].sort_values('start_time').drop_duplicates(['tags.candidate','tags.fold'],keep='last')
display(exp[['tags.candidate','tags.fold','metrics.areaUnderPR','metrics.areaUnderROC','metrics.prevalence','metrics.f1','metrics.fit_seconds']].sort_values(['tags.candidate','tags.fold']))
assert len(exp)==18
summary=exp.groupby('tags.candidate').agg(pr_media=('metrics.areaUnderPR','mean'),roc_media=('metrics.areaUnderROC','mean'),fit_medio_s=('metrics.fit_seconds','mean')).sort_values('pr_media',ascending=False)
display(summary)
if (E/'validation_comparison.png').exists(): display(Image(filename=str(E/'validation_comparison.png')))
display(json.loads((RUN/'decision.lock.json').read_text()))

## 8. Embeddings y comparación directa E4 frente a E1
all-MiniLM-L6-v2 congelado, revisión fijada, 384 dimensiones y normalización L2. mapInPandas procesa lotes de 32 con una copia por partición; se limita a una partición para no multiplicar memoria. El diccionario permite reutilizar textos y codificar nuevos con el mismo modelo. Representar texto nuevo no crea umbral histórico para un tipo nuevo. La tabla de descriptores no vistos usa umbral diagnóstico 0,5; sus cohortes pequeñas (192 y 81 casos) limitan la interpretación y no fijan la regla final.

In [ ]:
display(pd.read_csv(E/'e4_vs_e1.csv'))
display(pd.read_csv(E/'e4_unseen_descriptors.csv'))
emb=exp[exp['tags.family']=='E4']
display(emb[['tags.fold','metrics.embeddings_seconds','metrics.fit_seconds','metrics.unseen_descriptor_n','metrics.text_recovery_max_error','metrics.reload_max_error']])
assert (exp['metrics.reload_max_error']<1e-10).all()
assert (emb['metrics.text_recovery_max_error']<1e-6).all()
print('Se conserva el resultado de E4 aunque no mejore; tiempos de diccionario y clasificador separados.')

## 9. Escenarios de alerta y referencia trivial
Se muestran escenarios de validación, sin inventar límites operativos. La referencia trivial nunca alerta: su F1 es cero. La prevalencia sirve como referencia PR de un ordenamiento aleatorio; no es una curva estimada del modelo trivial.

In [ ]:
scenarios=pd.read_csv(E/'threshold_scenarios.csv');chosen=final['decision']['candidate']
display(scenarios[scenarios.candidate==chosen].groupby('threshold')[['precision','recall','f1','alert_fraction']].mean())
trivial=runs[(runs['tags.kind']=='trivial')&(runs.status=='FINISHED')]
display(trivial[['params.fold','metrics.n','metrics.prevalence','metrics.f1','metrics.alert_fraction']])

## 10. Benchmark y ajuste final
Se calienta una vez y se miden tres repeticiones de E1 fijo en dos tamaños. La inferencia se materializa sumando score, con caché DISK_ONLY explícita y caché de sistema operativo no controlada. No demuestra escalabilidad entre máquinas. Después de dos límites de disco y un ajuste provisional del 5%, se liberó almacenamiento y se completó el entrenamiento al 100%. La prueba 2024 ya se había observado con el modelo provisional; se mantuvo intacta la decisión fijada previamente y no se repitió tuning ni se escogió entre resultados de prueba.

In [ ]:
if EJECUTAR_DESDE_CERO:
    benchmark(spark,CONFIG,RUN)
    CONFIG.update(rdd_compress=True,final_partitions=64)
    # Nueva sesión para aplicar compresión RDD igual que la ejecución final.
    spark.stop(); spark=session(CONFIG); spark.sparkContext.setLogLevel('ERROR')
    environment(spark,CONFIG,RUN)
    final_fit(spark,CONFIG,RUN,full=True)
    # Si falla por recursos, seguir README para --sample-final con fraction=0.05; no cambiar la decisión.
    export_runs(RUN); spark.stop()
display(pd.read_csv(E/'benchmark_summary.csv'))
final=read('final_result.json');display(pd.DataFrame([final['metrics']]).T)
assert final['cohort']['train_used']==final['cohort']['train_eligible']-final['cohort']['train_no_threshold'], 'La entrega requiere 100% del entrenamiento elegible'
assert final['metrics']['train_n']==final['cohort']['train_used']
display(read('full_retry_protocol.json'))
display(read('baseline_all_alert.json'))
print('Train utilizado:',final['cohort']['train_used'],'| Elegible con umbral:',final['cohort']['train_eligible']-final['cohort']['train_no_threshold'])
if (E/'final_confusion.png').exists():display(Image(filename=str(E/'final_confusion.png')))
segments=pd.read_csv(E/'test_subgroups.csv')
display(segments[segments.group_variable.eq('borough')])

## 11. Procedencia del benchmark de fase 1
El PDF previo y el notebook contienen tiempos diferentes. El CSV de las tres corridas coincide con las salidas del notebook. Los horarios son compatibles con una ejecución posterior a generar el informe, pero no prueban por sí solos la causa. Se mantienen ambas procedencias.

In [ ]:
display(pd.read_csv(ROOT/'evidencia/benchmark_phase1_reconciliation.csv'))

## 12. Interpretación y límites
Las métricas y la fracción alertada se muestran arriba; su utilidad requiere validar capacidad operativa. No se cambia el umbral con esos resultados. La configuración seleccionada se recomienda bajo este protocolo y su muestra de desarrollo, no como garantía operativa. Las métricas se condicionan a cierres observados y tipos con umbral; persisten sesgo de selección, correcciones posteriores, incertidumbre de muestreo y corte de observación aproximado. Las diferencias por territorio y canal son descriptivas. No se usó 2024 para ajustar ni se atribuye escalabilidad distribuida entre máquinas a local[2].

**Uso de IA:** lectura, código, revisión temporal, depuración, pruebas e informe. Se exigió evidencia real y ausencia de métricas inventadas. Validación: hashes, Spark, conteos exactos, pruebas de bordes, conservación de filas, guardado/recarga, procesamiento textual y MLflow. El prompt original y las fuentes revisadas están conservados. Ver README, METODOLOGIA y el informe para reproducción y defensa.